# Task 3 - Kakuro as a CSP

Solves the two boards from the statement. The visualizer is started with `python main.py`.

In [1]:
from kakuro.puzzle import load_builtin, SolutionChecker
from kakuro.csp import KakuroCSP
from kakuro.solver import BacktrackingSolver, SolverConfig
from kakuro.evaluation import CorrectnessEvaluation, EffectivenessEvaluation, StabilityEvaluation, machine_description
from main import render_solution
print(machine_description())

Linux-6.18.44-fc-v470-x86_64-with-glibc2.39 | x86_64 x86_64 | Python 3.13.18


## Requirement 1 - variables, domains, constraints

In [2]:
p = load_builtin('Complex (9x9)')
csp = KakuroCSP(p)
print('white cells (variables):', len(p.cells))
print('runs (sum + all-different constraints):', len(p.runs) if hasattr(p, 'runs') else 'see csp')
print('domain of every variable before propagation: 1..9')

white cells (variables): 48
runs (sum + all-different constraints): 28
domain of every variable before propagation: 1..9


## Requirement 2 - solving both boards

In [3]:
for name in ('Example (5x5)', 'Complex (9x9)'):
    puzzle = load_builtin(name)
    result = BacktrackingSolver(KakuroCSP(puzzle)).solve()
    s = result.stats
    print(f'{name}: {result.status}, {result.seconds*1000:.2f} ms, {s.assignments} assignments, {s.backtracks} backtracks')
    print(render_solution(puzzle, result.solution))
    print('check:', SolutionChecker(puzzle).problems(result.solution) or 'all rules satisfied', '\n')

Example (5x5): solved, 0.65 ms, 12 assignments, 0 backtracks
# # # # #
# # 8 4 #
# 9 3 1 8
# 5 1 2 3
# # 9 7 #
check: all rules satisfied 

Complex (9x9): solved, 3.04 ms, 48 assignments, 0 backtracks
# # # # # # # # #
# # 5 9 8 # 1 2 #
# # 2 7 1 3 4 6 5
# 8 6 # # 1 2 8 9
# 9 8 # 3 2 8 9 #
# # 9 8 6 5 # 7 5
# 3 4 5 2 # # 4 2
# 9 3 7 1 6 8 5 #
# # 7 9 # 1 4 3 #
check: all rules satisfied 



### Correctness

In [4]:
print(CorrectnessEvaluation().run())

                                                 check |        puzzle |   result
-------------------------------------------------------+---------------+---------
              independent checker finds no broken rule | Example (5x5) |     True
               number of solutions (exhaustive search) | Example (5x5) |        1
             all configurations give the same solution | Example (5x5) |     True
   search state fully restored after exhaustive search | Example (5x5) |     True
boards with one clue raised are reported unsatisfiable | Example (5x5) |   4 of 4
              independent checker finds no broken rule | Complex (9x9) |     True
               number of solutions (exhaustive search) | Complex (9x9) |        1
             all configurations give the same solution | Complex (9x9) |     True
   search state fully restored after exhaustive search | Complex (9x9) |     True
boards with one clue raised are reported unsatisfiable | Complex (9x9) | 14 of 14


### Effectiveness of propagation and heuristics (median of 3 runs)

In [5]:
print(EffectivenessEvaluation(repeats=3, time_limit=20).run())

       puzzle |               configuration | status | median_ms | assignments | branch_points | backtracks | revisions | digits_pruned | max_depth | valid
--------------+-----------------------------+--------+-----------+-------------+---------------+------------+-----------+---------------+-----------+------
Example (5x5) |   none / static / ascending | solved |      6.09 |        1132 |          1132 |       1120 |         0 |             0 |        12 |  True
Example (5x5) |     fc / static / ascending | solved |      0.46 |          33 |             8 |         21 |        60 |           144 |        12 |  True
Example (5x5) |        fc / mrv / ascending | solved |      0.38 |          13 |             2 |          1 |        34 |           101 |        12 |  True
Example (5x5) | fc / mrv+degree / ascending | solved |      0.45 |          13 |             2 |          1 |        34 |           101 |        12 |  True
Example (5x5) |       fc / mrv+degree / lcv | solved |      0.54

### Stability (random tie-breaking, 10 seeds, 3 s limit)

In [6]:
print(StabilityEvaluation(seeds=10, time_limit=3).run())

       puzzle |               configuration | runs | solved | distinct_solutions | ms_mean | ms_std |  ms_max | assign_mean | assign_std | assign_max | backtrack_mean | backtrack_max
--------------+-----------------------------+------+--------+--------------------+---------+--------+---------+-------------+------------+------------+----------------+--------------
Example (5x5) |   none / static / ascending |   10 |     10 |                  1 |   22.32 |  28.14 |   86.66 |     2586.10 |    3295.67 |      10261 |        2574.10 |         10249
Example (5x5) |     fc / static / ascending |   10 |     10 |                  1 |    0.62 |   0.07 |    0.77 |          43 |      14.38 |         75 |             31 |            63
Example (5x5) |        fc / mrv / ascending |   10 |     10 |                  1 |    0.51 |   0.03 |    0.55 |       14.80 |       1.99 |         19 |           2.80 |             7
Example (5x5) | fc / mrv+degree / ascending |   10 |     10 |                  1 |   